# Exploratory Data Analysis: MLB Pitch Info

Just some early exploration of the available data to make sure this project is doable; if the features don't have enough data, we're cooked

TO DO:
Cut infrequent pitches (what even is an SV??)

## Imports and setup

In [9]:
import pandas as pd
import numpy as np
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
 
from pybaseball import statcast
from datetime import datetime, timedelta
 
print("Imports successful")

Imports successful


## Fetch pitch-by-pitch data from Baseball Savant

All the boring data stuff

In [10]:
# Define date range. Note to self: this needs to be able to be updated in the future easily.
start_date = "2026-03-25"
end_date = "2026-08-29"

print(f"Fetching statcast data from {start_date} to {end_date}...")
print("(This may take 30-60 seconds on first run. Data is cached after)")

# Fetch all pitch-by-pitch data
df_raw = statcast(start_dt=start_date, end_dt=end_date)
 
if df_raw is None or len(df_raw) == 0:
    print("WARNING: No data returned. Solutions:")
    print("  - Date range is within MLB season (Mar-Sept)")
    print("  - Internet connection")
    print("  - Try a different season")
    df_pitches = pd.DataFrame()
else:
    print(f"Fetched {len(df_raw)} pitches!")
    
    # Extract only rows with valid pitch data (drop non-pitch events)
    df_pitches = df_raw[df_raw['pitch_type'].notna()].copy()
    
    print(f"After filtering: {len(df_pitches)} valid pitches.")
    
    # Print available columns first
    print(f"\nAvailable columns:")
    print(df_pitches.columns.tolist())
    
    # Print a sample of key columns
    key_cols = ['pitcher', 'batter', 'pitch_type', 'balls', 'strikes', 'inning']
    available_key_cols = [c for c in key_cols if c in df_pitches.columns]
    print(f"\nFirst few rows (key columns):")
    print(df_pitches[available_key_cols].head(10))
 
# Stop here if no data
if len(df_pitches) == 0:
    print("\nCannot proceed without data. Exiting.")
    exit()

Fetching statcast data from 2026-03-25 to 2026-08-29...
(This may take 30-60 seconds on first run. Data is cached after)
This is a large query, it may take a moment to complete


100%|██████████| 158/158 [00:39<00:00,  3.98it/s]


Fetched 602712 pitches!
After filtering: 600414 valid pitches.

Available columns:
['pitch_type', 'game_date', 'release_speed', 'release_pos_x', 'release_pos_z', 'player_name', 'batter', 'pitcher', 'events', 'description', 'spin_dir', 'spin_rate_deprecated', 'break_angle_deprecated', 'break_length_deprecated', 'zone', 'des', 'game_type', 'stand', 'p_throws', 'home_team', 'away_team', 'type', 'hit_location', 'bb_type', 'balls', 'strikes', 'game_year', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'on_3b', 'on_2b', 'on_1b', 'outs_when_up', 'inning', 'inning_topbot', 'hc_x', 'hc_y', 'tfs_deprecated', 'tfs_zulu_deprecated', 'umpire', 'sv_id', 'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az', 'sz_top', 'sz_bot', 'hit_distance_sc', 'launch_speed', 'launch_angle', 'effective_speed', 'release_spin_rate', 'release_extension', 'game_pk', 'fielder_2', 'fielder_3', 'fielder_4', 'fielder_5', 'fielder_6', 'fielder_7', 'fielder_8', 'fielder_9', 'release_pos_y', 'estimated_ba_using_speedangle', 'estimated_woba_using_s

## Pitch type distribution

Looking for how many unique pitch types are thrown, their percentages, etc. This is nothing groundbreaking; if 4-seam fastball isn't the most common, my data's wrong. Or, we're in BenLand where everyone throws really disgusting sinkers as a baseline.

In [11]:
pitch_counts = df_pitches['pitch_type'].value_counts()
print(f"Unique pitch types: {len(pitch_counts)}")
print("\nPitch type counts:")
print(pitch_counts)
print("\nPitch type percentages:")
print((pitch_counts / len(df_pitches) * 100).round(2))

# What's the baseline if we always predict the most common pitch?
baseline_pitch = pitch_counts.index[0]
baseline_accuracy = pitch_counts.values[0] / len(df_pitches)
print(f"\nBaseline: always predict '{baseline_pitch}' with {baseline_accuracy:.1%} accuracy")

Unique pitch types: 17

Pitch type counts:
pitch_type
FF    183636
SI     99870
SL     79376
CH     67302
ST     50012
FC     47752
CU     38290
FS     19437
KC      9600
SV      2660
EP      1016
FA       648
FO       401
KN       247
CS       122
PO        34
UN        11
Name: count, dtype: int64

Pitch type percentages:
pitch_type
FF    30.58
SI    16.63
SL    13.22
CH    11.21
ST     8.33
FC     7.95
CU     6.38
FS     3.24
KC     1.60
SV     0.44
EP     0.17
FA     0.11
FO     0.07
KN     0.04
CS     0.02
PO     0.01
UN     0.00
Name: count, dtype: float64

Baseline: always predict 'FF' with 30.6% accuracy


## Pitcher-batter matchup sparsity

One of many of my concerns is regarding pitcher-batter matchups, which are counted to a very limited degree (obviously, a batter won't have that many ABs against a single pitcher). This is just a check to see if I have the data to introduce that as a feature in the model.

In [12]:
# For each pitcher-batter pair, count how many pitches we have
# pybaseball uses 'pitcher' and 'batter' as player IDs
df_pitches['pitcher_batter_pair'] = (
    df_pitches['pitcher'].fillna(-1).astype(int).astype(str) + '_' + 
    df_pitches['batter'].fillna(-1).astype(int).astype(str)
)

matchup_pitch_counts = df_pitches['pitcher_batter_pair'].value_counts()
print(f"\nTotal pitcher-batter ABs: {len(matchup_pitch_counts)}")
print(f"ABs with 1 pitch: {(matchup_pitch_counts == 1).sum()}")
print(f"ABs with 2-3 pitches: {((matchup_pitch_counts >= 2) & (matchup_pitch_counts <= 3)).sum()}")
print(f"ABs with 4-5 pitches: {((matchup_pitch_counts >= 4) & (matchup_pitch_counts <= 5)).sum()}")
print(f"ABs with 6+ pitches: {(matchup_pitch_counts >= 6).sum()}")

# Show a brief preview of the distribution of matchup pitch counts
print(f"\nMatchup distribution (first 20):")
print(matchup_pitch_counts.head(20))

# What % of all pitches come from "rich" matchups (3+)?
rich_pairs = matchup_pitch_counts[matchup_pitch_counts >= 4].index
pitches_in_rich = df_pitches[df_pitches['pitcher_batter_pair'].isin(rich_pairs)].shape[0]
print(f"\nPitches from ABs with 4+ history: {pitches_in_rich / len(df_pitches):.1%}")
print(f"  - Pitch history feature: {'VIABLE' if pitches_in_rich / len(df_pitches) > 0.3 else 'SPARSE'}")


Total pitcher-batter ABs: 79491
ABs with 1 pitch: 4169
ABs with 2-3 pitches: 13127
ABs with 4-5 pitches: 17480
ABs with 6+ pitches: 44715

Matchup distribution (first 20):
pitcher_batter_pair
702070_668885    73
656302_677800    65
669022_687263    59
693645_672960    54
668909_678246    54
800048_805367    52
677958_670623    52
695505_664040    52
672282_679822    52
693645_677800    51
656302_680776    51
657746_695600    50
694738_672515    50
694738_672695    49
668909_803011    49
554430_669364    49
656302_575929    49
656876_672960    49
680694_700250    49
656302_678882    49
Name: count, dtype: int64

Pitches from ABs with 4+ history: 93.7%
  - Pitch history feature: VIABLE


## Count distribution across game states

Looking at the distribution of pitch count data (e.g. 0-2 pitches, 3-1, etc.). This targets another desired feature: pitch variance influenced by the count.

In [13]:
# Create a 'count' state (balls-strikes). You'd think they'd have this recorded... or I'm blind
df_pitches['count_state'] = (
    df_pitches['balls'].fillna(0).astype(int).astype(str) + '-' + 
    df_pitches['strikes'].fillna(0).astype(int).astype(str)
)

# Starting from 0-0 might sound stupid but I think it makes sense to track opening pitches.
count_dist = df_pitches['count_state'].value_counts().sort_index()
print(f"\nCount state distribution (balls-strikes):")
print(count_dist)
print("\nPercentages:")
print((count_dist / len(df_pitches) * 100).round(1))

# Check if any count is severely underrepresented
print(f"\nCount states with <2% of pitches:")
underrep = (count_dist / len(df_pitches) < 0.02)
if underrep.any():
    print(count_dist[underrep])
else:
    print("  None, data well distributed!")


Count state distribution (balls-strikes):
count_state
0-0    154334
0-1     77262
0-2     40624
1-0     59126
1-1     60058
1-2     58446
2-0     20073
2-1     31096
2-2     49593
3-0      6449
3-1     13248
3-2     30105
Name: count, dtype: int64

Percentages:
count_state
0-0    25.7
0-1    12.9
0-2     6.8
1-0     9.8
1-1    10.0
1-2     9.7
2-0     3.3
2-1     5.2
2-2     8.3
3-0     1.1
3-1     2.2
3-2     5.0
Name: count, dtype: float64

Count states with <2% of pitches:
count_state
3-0    6449
Name: count, dtype: int64


## Inning and workload distribution

Looking into workload and how that affects pitch decisions. I honestly don't really know how much it will, but who knows? Not like I've ever pitched an MLB game

In [14]:
inning_dist = df_pitches['inning'].value_counts().sort_index()
print(f"\nPitches per inning:")
print(inning_dist)

# Workload: cumulative pitch count per pitcher per game
# pybaseball uses 'game_date' or similar; find the game identifier column
df_pitches['game_id'] = df_pitches['game_date'].astype(str) + '_' + df_pitches['pitcher'].fillna(-1).astype(int).astype(str)
df_pitches = df_pitches.sort_values(['game_date', 'pitcher', 'inning', 'pitch_number'])
df_pitches['pitch_count_in_game'] = (
    df_pitches.groupby('game_id').cumcount() + 1
)
 
print(f"\nPitch count ranges (per pitcher per game):")
print(f"  - Min: {df_pitches['pitch_count_in_game'].min()}")
print(f"  - Max: {df_pitches['pitch_count_in_game'].max()}")
print(f"  - Mean: {df_pitches['pitch_count_in_game'].mean():.1f}")

# Bucket into workload phases
def bucket_workload(pitch_count):
    if pitch_count <= 20:
        return '0-20'
    elif pitch_count <= 50:
        return '21-50'
    elif pitch_count <= 80:
        return '51-80'
    else:
        return '81+'
 
df_pitches['workload_bucket'] = df_pitches['pitch_count_in_game'].apply(bucket_workload)
workload_dist = df_pitches['workload_bucket'].value_counts()
print(f"\nPitches by workload bucket:")
print(workload_dist)
print("\nPercentages:")
print((workload_dist / len(df_pitches) * 100).round(1))


Pitches per inning:
inning
1     69524
2     66968
3     67159
4     66855
5     66960
6     67673
7     68156
8     68807
9     51480
10     5092
11     1205
12      460
13       75
Name: count, dtype: Int64

Pitch count ranges (per pitcher per game):
  - Min: 1
  - Max: 120
  - Mean: 31.4

Pitches by workload bucket:
workload_bucket
0-20     286060
21-50    163438
51-80    113039
81+       37877
Name: count, dtype: int64

Percentages:
workload_bucket
0-20     47.6
21-50    27.2
51-80    18.8
81+       6.3
Name: count, dtype: float64


## Export

Clearly I was overthinking the data problem

In [15]:
output_path = 'mlb_pitches_eda.csv'
df_pitches.to_csv(output_path, index=False)
print(f"\nSaved processed pitch data to {output_path}")
print(f"  Columns: {list(df_pitches.columns)}")


Saved processed pitch data to mlb_pitches_eda.csv
  Columns: ['pitch_type', 'game_date', 'release_speed', 'release_pos_x', 'release_pos_z', 'player_name', 'batter', 'pitcher', 'events', 'description', 'spin_dir', 'spin_rate_deprecated', 'break_angle_deprecated', 'break_length_deprecated', 'zone', 'des', 'game_type', 'stand', 'p_throws', 'home_team', 'away_team', 'type', 'hit_location', 'bb_type', 'balls', 'strikes', 'game_year', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'on_3b', 'on_2b', 'on_1b', 'outs_when_up', 'inning', 'inning_topbot', 'hc_x', 'hc_y', 'tfs_deprecated', 'tfs_zulu_deprecated', 'umpire', 'sv_id', 'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az', 'sz_top', 'sz_bot', 'hit_distance_sc', 'launch_speed', 'launch_angle', 'effective_speed', 'release_spin_rate', 'release_extension', 'game_pk', 'fielder_2', 'fielder_3', 'fielder_4', 'fielder_5', 'fielder_6', 'fielder_7', 'fielder_8', 'fielder_9', 'release_pos_y', 'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle', 'woba_val